In [1]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset

class CandleDataset(Dataset):
    def __init__(self, df, feature_cols, window=60):
        self.features = df[feature_cols].values.astype(np.float32)
        self.labels = df["label"].values.astype(np.int64)
        self.window = window

    def __len__(self):
        return max(0, len(self.features) - self.window)

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window]          # (window, n_features)
        y = self.labels[idx + self.window - 1]                # лейбл на конец окна
        return torch.tensor(x), torch.tensor(y)

In [2]:
from model import prepare_features_and_labels

class WindowExampleDataset(Dataset):
    """Каждый df целиком = один пример: window строк фичей -> 1 label"""
    def __init__(self, dfs, feature_cols, window=120, horizon=5, flat_threshold=0.001):
        self.examples = []
        
        for df in dfs:
            #df_feat = prepare_features_and_labels(df,horizon=horizon)
            #df_feat = df_feat.dropna().reset_index(drop=True)
            
            if len(df) < window:
                print(len(df))
                continue  # пропускаем, если после dropna не хватает данных на полный контекст
            
            context = df.iloc[:window]
            #print(len(df))
            # цена в конце контекста и цена в конце горизонта (последняя из 5 будущих свечей)
            close_at_context_end = df["close"].iloc[window - 1]
            close_at_horizon_end = df["close"].iloc[window + horizon - 1]
            future_return = close_at_horizon_end - close_at_context_end
            
            if future_return > flat_threshold:
                label = 2
            elif future_return < -flat_threshold:
                label = 0
            else:
                label = 1
            
            x = context[feature_cols].values.astype(np.float32)
            self.examples.append((x, label))
        
        print(f"Valid examples: {len(self.examples)} / {len(dfs)}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        x, y = self.examples[idx]
        return torch.tensor(x), torch.tensor(y, dtype=torch.int64)

In [3]:
print(torch.__version__)  
print(torch.cuda.is_available())

2.13.0+cu132
True


In [4]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from model import LSTMClassifier
import torch.nn as nn

def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, device="cuda"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # валидация
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                logits = model(x)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y.numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"Epoch {epoch+1}: train_loss={train_loss/len(train_loader):.4f}, val_acc={acc:.4f}, val_f1={f1:.4f}")

    return model

In [5]:
import pandas as pd
from generator import loadDataSet
from model import prepare_features_and_labels

# loading train data
dfs_raw = loadDataSet('data/synthetic/0046_002_4_5')

dfs = [0]*len(dfs_raw)
print(len(dfs_raw))
for i,df in enumerate(dfs_raw):
    dfs[i] = prepare_features_and_labels(df, horizon=5, flat_threshold=0.001)
    


27963


In [ ]:
import pandas as pd
from model import prepare_features_and_labels, feature_cols



# train/val split
split_idx = int(len(dfs) * 0.8)
train_dfs, val_dfs = dfs[:split_idx], dfs[split_idx:]


train_ds = WindowExampleDataset(train_dfs, feature_cols, window=120, horizon=5, flat_threshold=0.001)
val_ds = WindowExampleDataset(val_dfs, feature_cols, window=120, horizon=5, flat_threshold=0.001)


train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=True)

model = LSTMClassifier(n_features=len(feature_cols), num_layers=2, num_classes=3)
print(f"Learning on {'cuda' if torch.cuda.is_available() else 'cpu'}")
model = train_model(
    model, 
    train_loader, 
    val_loader,
    epochs=400,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

Valid examples: 22370 / 22370
Valid examples: 5593 / 5593
Learning on cuda
Epoch 1: train_loss=0.7556, val_acc=0.4920, val_f1=0.2199
Epoch 2: train_loss=0.7356, val_acc=0.5013, val_f1=0.2226
Epoch 3: train_loss=0.7354, val_acc=0.5013, val_f1=0.2226
Epoch 4: train_loss=0.7352, val_acc=0.4920, val_f1=0.2199
Epoch 5: train_loss=0.7356, val_acc=0.5013, val_f1=0.2226
Epoch 6: train_loss=0.7349, val_acc=0.4920, val_f1=0.2199
Epoch 7: train_loss=0.7350, val_acc=0.5013, val_f1=0.2226
Epoch 8: train_loss=0.7348, val_acc=0.4920, val_f1=0.2199
Epoch 9: train_loss=0.7344, val_acc=0.5013, val_f1=0.2226
Epoch 10: train_loss=0.7345, val_acc=0.4920, val_f1=0.2199
Epoch 11: train_loss=0.7345, val_acc=0.5013, val_f1=0.2226
Epoch 12: train_loss=0.7346, val_acc=0.4920, val_f1=0.2199
Epoch 13: train_loss=0.7348, val_acc=0.5013, val_f1=0.2226
Epoch 14: train_loss=0.7344, val_acc=0.5013, val_f1=0.2226
Epoch 15: train_loss=0.7344, val_acc=0.4920, val_f1=0.2199
Epoch 16: train_loss=0.7347, val_acc=0.4920, val_

In [ ]:
torch.save(model.state_dict(), "models/lstm_3360000_400_0046_23.pt")